In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv("DateFruit_Dataset.csv")
#df.isnull().sum()
X=df.drop("Class",axis=1)
y=df["Class"]
df["Class"].unique()


array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [3]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le=LabelEncoder()

y=le.fit_transform(y)

In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
sc=StandardScaler()
X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.fit_transform(X_test)

In [6]:
#ANN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [7]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)


In [8]:
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)


In [9]:
#build model
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self).__init__()
    self.model=nn.Sequential(
        #1st
        nn.Linear(X.shape[1],64),
        nn.ReLU(),
        #2nd
        nn.Linear(64,64),
        nn.ReLU(),
        #o/p
        nn.Linear(64,7)

    )
  def forward(self,x):
    return self.model(x)




In [10]:
model=ANN()
#loss optim
criteria=nn.CrossEntropyLoss()
optimizers=optim.Adam(model.parameters())


In [11]:
#training no validation
epochs=100
for epoch in range(epochs):
  model.train()
  running_loss=0.0
  for xb,yb in train_loader:
    optimizers.zero_grad()
    outputs=model(xb)
    loss=criteria(outputs,yb)
    loss.backward()
    optimizers.step()
    running_loss+=loss

  epoch_train_loss=running_loss/len(train_loader)

In [12]:
model.eval()
total=0
correct=0
with torch.no_grad():
  for xb,yb in test_loader:
    outputs=model(xb)
    _,predicted=torch.max(outputs,1)

    correct+=(predicted==yb).sum().item()
    total+=yb.size(0)
print(f"acc:{correct/total}")


acc:0.9333333333333333
